# GNN Candidate Screening Workflow

This notebook runs inference over a GNN dataset to predict activation energies and rank materials for further MACE evaluation.

## 1. Configuration

In [ ]:
import os
import sys
import torch
import pandas as pd
import libraries.screen_candidates as sc
import importlib

# Reload to ensure latest changes
importlib.reload(sc)

# --- SETTINGS ---
MODEL_DIR = "models/all-energies/results_20260510_125735"
DATASET_PATH = "models/all-energies/dataset.pt"
STD_PARAMS_PATH = "models/all-energies/standardized_parameters.json"

# --- RANKING METHOD ---
# Set to None for single target ranking, or a dict for weighted sum
# Example: {"E_3D": 0.6, "E_2D": 0.3, "E_1D": 0.1}
WEIGHTS = {"E_3D": 0.6, "E_2D": 0.3, "E_1D": 0.1}
TARGET = "E_3D"           # Ranking target (used only if WEIGHTS is None)

TOP_N = 10                # Number of candidates to export to candidates.txt
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CANDIDATES_TXT = "candidates.txt"
PREDICTIONS_CSV = "candidate_predictions.csv"

print(f"Using device: {DEVICE}")

## 2. Load Model

In [ ]:
print(f"Loading model from {MODEL_DIR}...")
model, model_params = sc.load_screening_model(MODEL_DIR, device=DEVICE)
print("Model loaded successfully.")

## 3. Load Dataset

In [ ]:
print(f"Loading dataset from {DATASET_PATH}...")
dataset_std, std_params = sc.load_screening_dataset(DATASET_PATH, STD_PARAMS_PATH, model_params)
print(f"Loaded {len(dataset_std)} samples.")

## 4. Run Inference

In [ ]:
print("Running inference over all samples...")
results = sc.run_inference(
    model=model, 
    dataset=dataset_std, 
    device=DEVICE, 
    std_params=std_params, 
    model_params=model_params, 
    batch_size=BATCH_SIZE
)
print(f"Inference complete. Processed {len(results)} materials.")

## 5. Rank Candidates

In [ ]:
print("Ranking candidates...")
ranked_df = sc.rank_candidates(results, target=TARGET, weights=WEIGHTS)
print("Ranking complete.")
display(ranked_df.head(TOP_N))

## 6. Visualization

In [ ]:
# Plot distribution of predictions
sc.plot_energy_distributions(results)

# Plot top candidates bar chart
sc.plot_top_candidates(ranked_df, top_n=TOP_N)

## 7. Export Outputs

In [ ]:
# Export candidates.txt for run-MACE.ipynb
sc.write_candidates_txt(ranked_df, TOP_N, CANDIDATES_TXT)

# Export full CSV for analysis
sc.write_predictions_csv(ranked_df, PREDICTIONS_CSV)